In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from collections import defaultdict, deque
import math

# =======================================================
# 1. LOAD PPI AND DPI FILES
# =======================================================
ppi = pd.read_csv("/content/drive/MyDrive/FinalWork/SIGNOR_ppi_GO_STRING_with_IntAct.csv")
#dpi = pd.read_csv("/content/drive/MyDrive/FinalWork/dpi_filtered_clean.csv") old file
dpi = pd.read_csv("/content/drive/MyDrive/evidti_dpi_filtered_clean.csv")

# Required columns:
# ppi: IDA, IDB, confidence_score, GO_norm
# dpi: drugbank_id, uniprot_id

lambda_go = 1.0   # GO contribution multiplier
alpha = 0.85      # PPR continuation probability
eps   = 1e-5      # residual threshold (tune for speed vs accuracy)
# =======================================================
# 2. COLLAPSE DUPLICATE PPIs (IDA, IDB)
#    We keep:
#       - max confidence_score (most reliable)
#       - max GO_norm (most functionally similar)
#    This avoids artificial inflation from identical repeated rows.
# =======================================================
ppi_reduced = (
    ppi[["IDA", "IDB", "confidence_score", "GO_norm"]]
      .fillna({"confidence_score": 0.0, "GO_norm": 0.0})
      .groupby(["IDA", "IDB"], as_index=False)
      .agg({
          "confidence_score": "max",
          "GO_norm": "max"
      })
)

print(f"Original PPIs: {len(ppi)}, Unique (IDA,IDB): {len(ppi_reduced)}")


# =======================================================
# 3. BUILD DIRECTED GRAPH WITH RAW WEIGHTS
#    w_uv = conf * (1 + lambda_go * GO_norm)
# =======================================================
G = nx.DiGraph()
rows_raw = []

for _, r in ppi_reduced.iterrows():
    a = r["IDA"]               # UniProt A
    b = r["IDB"]               # UniProt B
    conf = float(r["confidence_score"])
    go   = float(r["GO_norm"])

    w = conf * (1.0 + lambda_go * go)

    if w < 0:
        w = 0.0

    G.add_edge(a, b, weight=w)
    rows_raw.append([a, b, w])

# Save raw weights (diagnostic)
trans_raw_df = pd.DataFrame(rows_raw, columns=["source", "target", "raw_weight"])
trans_raw_df.to_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/transition_matrix_raw_dedup.csv", index=False)


# =======================================================
# 4. NORMALIZE TO TRANSITION PROBABILITIES
#    P(u→v) = w_uv / sum_w(w_uw)
#    Dangling nodes = nodes with no outgoing probability (sum == 0)
# =======================================================
out_sums = defaultdict(float)
for u, v, data in G.edges(data=True):
    out_sums[u] += data["weight"]

for u, v, data in G.edges(data=True):
    if out_sums[u] > 0:
        data["p_uv"] = data["weight"] / out_sums[u]
    else:
        data["p_uv"] = 0.0   # will be treated as dangling in PPR

# Save normalized transitions
trans_norm = [[u, v, d["p_uv"]] for u, v, d in G.edges(data=True)]
pd.DataFrame(trans_norm, columns=["source", "target", "p_uv"]).to_csv(
    "/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/transition_matrix_normalized_dedup.csv", index=False
)

print("Saved transition_matrix_raw_dedup.csv and transition_matrix_normalized_dedup.csv")




/tmp/ipykernel_368/1630425042.py:10: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  ppi = pd.read_csv("/content/drive/MyDrive/FinalWork/SIGNOR_ppi_GO_STRING_with_IntAct.csv")


Original PPIs: 18950, Unique (IDA,IDB): 14130
Saved transition_matrix_raw_dedup.csv and transition_matrix_normalized_dedup.csv


In [ ]:
# =========================================================
# 0. IMPORTS
# =========================================================
import pandas as pd
from collections import defaultdict, deque

# =========================================================
# 1. LOAD FILES (YOUR ACTUAL FILES)
# =========================================================

trans = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/transition_matrix_normalized_dedup.csv")
#dpi   = pd.read_csv("/content/drive/MyDrive/FinalWork/dpi_filtered_clean.csv")
dpi = pd.read_csv("/content/drive/MyDrive/evidti_dpi_filtered_clean.csv")


# Keep only needed columns
dpi = dpi[["drugbank_id", "uniprot_id"]]
dpi.columns = ["drugbank_id", "protein_id"]

print("Files loaded")

# =========================================================
# 2. BUILD GRAPH (FAST STRUCTURE)
# =========================================================

adj = defaultdict(list)

for _, r in trans.iterrows():
    u = r["source"]
    v = r["target"]
    p = r["p_uv"]

    adj[u].append((v, p))

print("Graph built")

# =========================================================
# 3. BUILD DRUG → TARGETS MAP
# =========================================================

drug_targets = defaultdict(list)

for _, r in dpi.iterrows():
    drug = r["drugbank_id"]
    protein = r["protein_id"]
    drug_targets[drug].append(protein)

print("Drug-target mapping ready:", len(drug_targets))

# =========================================================
# 4. PERSONALIZATION VECTOR
# =========================================================

def build_personalization(drug, targets, lambda_=0.7):
    v = defaultdict(float)

    v[drug] = lambda_

    if len(targets) > 0:
        share = (1 - lambda_) / len(targets)
        for t in targets:
            v[t] += share

    return v

# =========================================================
# 5. FAST PUSH-BASED PPR
# =========================================================

def push_ppr_fast(adj, v, alpha=0.85, eps=1e-4):

    p = defaultdict(float)
    r = defaultdict(float)

    # Initialize residual
    for k, val in v.items():
        r[k] = val

    queue = deque()
    in_queue = set()

    # Initialize queue
    for k in r:
        if r[k] > eps:
            queue.append(k)
            in_queue.add(k)

    while queue:
        u = queue.popleft()
        in_queue.remove(u)

        ru = r[u]
        if ru < eps:
            continue

        r[u] = 0.0

        # Accumulate PPR
        p[u] += (1 - alpha) * ru
        remaining = alpha * ru

        neighbors = adj.get(u)

        # Normal node
        if neighbors:
            for v_node, prob in neighbors:
                inc = remaining * prob
                if inc == 0:
                    continue

                new_val = r[v_node] + inc
                r[v_node] = new_val

                if new_val > eps and v_node not in in_queue:
                    queue.append(v_node)
                    in_queue.add(v_node)

        # Dangling node
        else:
            for k, vk in v.items():
                inc = remaining * vk
                if inc == 0:
                    continue

                new_val = r[k] + inc
                r[k] = new_val

                if new_val > eps and k not in in_queue:
                    queue.append(k)
                    in_queue.add(k)

    # Normalize
    Z = sum(p.values())
    if Z > 0:
        inv_Z = 1.0 / Z
        for k in p:
            p[k] *= inv_Z

    return dict(p)

# =========================================================
# 6. RUN PPR FOR ALL DRUGS
# =========================================================

output_rows = []

total = len(drug_targets)

for idx, drug in enumerate(drug_targets.keys()):

    targets = drug_targets[drug]

    # Build personalization
    v = build_personalization(drug, targets, lambda_=0.7)

    # Run PPR
    ppr = push_ppr_fast(adj, v, alpha=0.85, eps=1e-4)

    # Save results
    for node, score in ppr.items():
        output_rows.append([drug, node, score])

    # Progress print
    if idx % 50 == 0:
        print(f"Processed {idx}/{total} drugs")

print("PPR finished")

# =========================================================
# 7. SAVE RESULTS
# =========================================================

ppr_df = pd.DataFrame(output_rows, columns=["drugbank_id", "node", "ppr_score"])

ppr_df.to_csv(
    "/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/ppr_results_personalized_FAST_0.7.csv",
    index=False
)

print("DONE: Results saved")

Files loaded
Graph built
Drug-target mapping ready: 5830
Processed 0/5830 drugs
Processed 50/5830 drugs
Processed 100/5830 drugs
Processed 150/5830 drugs
Processed 200/5830 drugs
Processed 250/5830 drugs
Processed 300/5830 drugs
Processed 350/5830 drugs
Processed 400/5830 drugs
Processed 450/5830 drugs
Processed 500/5830 drugs
Processed 550/5830 drugs
Processed 600/5830 drugs
Processed 650/5830 drugs
Processed 700/5830 drugs
Processed 750/5830 drugs
Processed 800/5830 drugs
Processed 850/5830 drugs
Processed 900/5830 drugs
Processed 950/5830 drugs
Processed 1000/5830 drugs
Processed 1050/5830 drugs
Processed 1100/5830 drugs
Processed 1150/5830 drugs
Processed 1200/5830 drugs
Processed 1250/5830 drugs
Processed 1300/5830 drugs
Processed 1350/5830 drugs
Processed 1400/5830 drugs
Processed 1450/5830 drugs
Processed 1500/5830 drugs
Processed 1550/5830 drugs
Processed 1600/5830 drugs
Processed 1650/5830 drugs
Processed 1700/5830 drugs
Processed 1750/5830 drugs
Processed 1800/5830 drugs
Proc

In [ ]:
ppr_df.groupby("drugbank_id")["ppr_score"].sum().head()

,ppr_score
drugbank_id,
DB00093,1.0
DB00104,1.0
DB00114,1.0
DB00117,1.0
DB00118,1.0
